
# Three-regime MS-AA vs PCA simulation benchmark

This notebook implements the final simulation design for the manuscript using the same `Subject`, `multi_subject_aa(...)`, spatial MS-AA orientation, and `sXC @ S` reconstruction logic as the empirical analysis.

It compares three generative regimes:

1. **PCA-favorable control:** orthogonal Gaussian latent factors.
2. **AA-favorable control:** sparse convex mixtures of nonorthogonal extremal motifs.
3. **Brain-inspired metastable dynamics:** recurring, overlapping network configurations with temporal persistence, gradual transitions, low-level coactivation, subject variability, and measurement noise.

The central question is not whether one method wins universally. Instead:

> Does each method behave as expected when its own geometry is truly present, and under a brain-inspired regime that is not generated exactly by either model, which representation more directly recovers the underlying network configurations?

For every regime, the notebook measures reconstruction error, one-to-one latent-pattern recovery, subnetwork recovery, PCA-subspace recovery, purity/selectivity, and off-target mixing.

Start with `RUN_MODE = "quick"`. If the complete notebook behaves correctly, switch to `RUN_MODE = "paper"`.


In [ ]:

RUN_MODE = "quick"   # "quick" or "paper"
RNG_SEED = 42

HELPERS_DIR = "helpers"
OUTPUT_DIR = "simulation_three_regime_outputs"
ANALYSIS_TYPE = "spatial"

MSAA_OPTS_BASE = dict(
    maxiter=200,
    conv_crit=1e-6,
    fix_var_iter=5,
    heteroscedastic=False,
    use_gpu=False,
    rngSEED=RNG_SEED,
)

N_NETWORKS = 6
SUBNETWORKS_PER_NETWORK = 2
NODES_PER_SUBNETWORK = 10
N_TRUE_MOTIFS = 6

BASE_NOISE_SD = 0.25
BASE_MULTIPLICATIVE_SD = 0.08
BASE_ADDITIVE_SD = 0.04

DIRICHLET_ALPHA = 0.10

# Brain-inspired metastable settings
MEAN_DWELL = 12
TRANSITION_WIDTH = 4
SECONDARY_WEIGHT = 0.25
AR1_RHO = 0.85

NOISE_LEVELS = [0.10, 0.25, 0.40]
HETEROGENEITY_LEVELS = [0.0, 1.0, 1.5]
K_LEVELS = [4, 6, 8]

if RUN_MODE == "quick":
    N_SUBJECTS = 10
    T_PER_SUBJECT = 120
    N_REPS_CONTROL = 2
    N_REPS_METASTABLE = 3
    N_REPS_ROBUST = 1
elif RUN_MODE == "paper":
    N_SUBJECTS = 16
    T_PER_SUBJECT = 180
    N_REPS_CONTROL = 20
    N_REPS_METASTABLE = 20
    N_REPS_ROBUST = 10
else:
    raise ValueError("RUN_MODE must be 'quick' or 'paper'")

print("RUN_MODE:", RUN_MODE)
print("Subjects:", N_SUBJECTS)
print("Time points per subject:", T_PER_SUBJECT)


In [ ]:

%matplotlib inline

import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import linear_sum_assignment
from sklearn.decomposition import PCA

sys.path.append(HELPERS_DIR)
from MultiSubject_AA import Subject, multi_subject_aa

OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_CSV = OUTPUT_DIR / f"three_regime_results_{RUN_MODE}.csv"
print("Checkpoint:", RESULTS_CSV)


## 1. MS-AA helpers matching the empirical pipeline

In [ ]:

def coerce_TxV(x):
    x = np.asarray(x, dtype=float)
    if x.ndim != 2:
        raise ValueError(f"Expected 2D T x V matrix, got {x.shape}")
    x = x - x.mean(axis=0, keepdims=True)
    return x.astype(np.float32, copy=False)


def orient_subject_matrix(X_raw, analysis_type):
    Xtv = coerce_TxV(X_raw)
    return Xtv if analysis_type == "spatial" else Xtv.T


def build_subject_objects(subject_list, analysis_type):
    return [
        Subject(
            X=orient_subject_matrix(x, analysis_type),
            sX=orient_subject_matrix(x, analysis_type),
        )
        for x in subject_list
    ]


def fit_msaa(subject_list, K, analysis_type="spatial", seed=RNG_SEED, verbose=False):
    opts = dict(MSAA_OPTS_BASE)
    opts["rngSEED"] = int(seed)

    subj_objs = build_subject_objects(subject_list, analysis_type)

    t0 = time.time()
    results_subj, C, cost_fun, varexpl, secs = multi_subject_aa(
        subj_objs,
        noc=int(K),
        opts=opts,
    )
    elapsed = time.time() - t0

    if verbose:
        print(f"[MS-AA] K={K} | VarExpl={varexpl*100:.2f}% | {elapsed:.1f}s")
        print("subj0 sXC:", np.asarray(results_subj[0]["sXC"]).shape)
        print("subj0 S:", np.asarray(results_subj[0]["S"]).shape)

    return {
        "results_subj": results_subj,
        "C": C,
        "cost_fun": cost_fun,
        "varexpl": varexpl,
        "secs": secs,
        "elapsed": elapsed,
        "K": int(K),
    }


def reconstruct_from_archetypes(sub, analysis_type="spatial"):
    sXC = np.asarray(sub["sXC"], dtype=float)
    S = np.asarray(sub["S"], dtype=float)
    return sXC @ S if analysis_type == "spatial" else (sXC @ S).T


## 2. Shared network motifs

In [ ]:

def make_overlapping_motifs():
    n_subnetworks = N_NETWORKS * SUBNETWORKS_PER_NETWORK
    V = n_subnetworks * NODES_PER_SUBNETWORK

    motifs = np.full((N_TRUE_MOTIFS, V), 0.05, dtype=float)

    motif_subnetworks = [
        [0, 3, 8],
        [1, 3, 9],
        [2, 5, 10],
        [2, 4, 11],
        [0, 6, 7],
        [1, 6, 8],
    ]

    for k, active in enumerate(motif_subnetworks):
        for subnet in active:
            lo = subnet * NODES_PER_SUBNETWORK
            hi = lo + NODES_PER_SUBNETWORK
            motifs[k, lo:hi] = 1.0

    return motifs


TRUE_MOTIFS_RAW = make_overlapping_motifs()
V = TRUE_MOTIFS_RAW.shape[1]
print("Ground-truth motif shape:", TRUE_MOTIFS_RAW.shape)


In [ ]:

def plot_pattern_matrix(patterns, title, ylabel="Pattern"):
    fig, ax = plt.subplots(figsize=(12, 4))
    im = ax.imshow(patterns, aspect="auto")

    for x in range(NODES_PER_SUBNETWORK, patterns.shape[1], NODES_PER_SUBNETWORK):
        ax.axvline(x - 0.5, linewidth=0.4, alpha=0.35)

    network_width = SUBNETWORKS_PER_NETWORK * NODES_PER_SUBNETWORK
    for x in range(network_width, patterns.shape[1], network_width):
        ax.axvline(x - 0.5, linewidth=1.2)

    ax.set_xlabel("Nodes grouped by network → subnetwork")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, shrink=0.8)
    plt.show()


plot_pattern_matrix(
    TRUE_MOTIFS_RAW,
    "Ground-truth overlapping distributed network motifs",
    ylabel="True motif",
)


## 3. Evaluation metrics

In [ ]:

def normalize_across_subjects(subject_list):
    subjects = [np.asarray(x, dtype=float) for x in subject_list]
    stacked = np.vstack(subjects)

    mu = stacked.mean(axis=0, keepdims=True)
    sd = stacked.std(axis=0, keepdims=True) + 1e-8

    normalized = [
        ((x - mu) / sd).astype(np.float32, copy=False)
        for x in subjects
    ]
    return normalized, mu, sd


def pattern_corr_matrix(A, B, absolute=False):
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)

    A0 = A - A.mean(axis=1, keepdims=True)
    B0 = B - B.mean(axis=1, keepdims=True)

    A0 /= np.linalg.norm(A0, axis=1, keepdims=True) + 1e-12
    B0 /= np.linalg.norm(B0, axis=1, keepdims=True) + 1e-12

    C = A0 @ B0.T
    return np.abs(C) if absolute else C


def optimal_match(true_patterns, recovered_patterns, absolute=False):
    C = pattern_corr_matrix(true_patterns, recovered_patterns, absolute=absolute)
    true_idx, recovered_idx = linear_sum_assignment(-C)
    return {
        "similarity": C,
        "true_idx": true_idx,
        "recovered_idx": recovered_idx,
        "matched": C[true_idx, recovered_idx],
    }


def collapse_to_subnetworks(patterns):
    patterns = np.asarray(patterns, dtype=float)
    n_subnetworks = N_NETWORKS * SUBNETWORKS_PER_NETWORK
    out = np.zeros((patterns.shape[0], n_subnetworks))

    for subnet in range(n_subnetworks):
        lo = subnet * NODES_PER_SUBNETWORK
        hi = lo + NODES_PER_SUBNETWORK
        out[:, subnet] = patterns[:, lo:hi].mean(axis=1)

    return out


def dimension_mixing_metrics(true_patterns, recovered_patterns):
    C = pattern_corr_matrix(recovered_patterns, true_patterns, absolute=True)

    squared = C ** 2
    probs = squared / (squared.sum(axis=1, keepdims=True) + 1e-12)

    entropy = -np.sum(probs * np.log(probs + 1e-12), axis=1)
    entropy /= np.log(true_patterns.shape[0])
    purity = 1.0 - entropy

    sorted_corr = np.sort(C, axis=1)[:, ::-1]
    top1 = sorted_corr[:, 0]
    top2 = sorted_corr[:, 1]

    return {
        "purity": float(np.mean(purity)),
        "selectivity": float(np.mean(top1 - top2)),
        "mixing_ratio": float(np.mean(top2 / (top1 + 1e-12))),
    }


## 4. Unified PCA/MS-AA evaluator

In [ ]:

def evaluate_methods(
    subject_data,
    true_patterns_raw,
    K_fit,
    seed,
    regime,
    rep,
    extra=None,
    verbose=False,
):
    subjects, mu, sd = normalize_across_subjects(subject_data)
    true_patterns = (true_patterns_raw - mu) / sd
    X_stack = np.vstack(subjects)

    # MS-AA
    msaa = fit_msaa(
        subjects,
        K=K_fit,
        analysis_type="spatial",
        seed=seed,
        verbose=verbose,
    )

    S_group = np.mean(
        np.stack(
            [np.asarray(sub["S"], dtype=float) for sub in msaa["results_subj"]],
            axis=0,
        ),
        axis=0,
    )

    Xhat_aa = np.vstack([
        reconstruct_from_archetypes(sub, "spatial")
        for sub in msaa["results_subj"]
    ])

    aa_mse = np.mean((X_stack - Xhat_aa) ** 2)

    # PCA
    pca = PCA(n_components=K_fit, random_state=seed)
    scores = pca.fit_transform(X_stack)
    Xhat_pca = pca.inverse_transform(scores)
    pca_mse = np.mean((X_stack - Xhat_pca) ** 2)

    # Direct pattern recovery
    aa_match = optimal_match(true_patterns, S_group, absolute=False)
    pca_match = optimal_match(true_patterns, pca.components_, absolute=True)

    # Subnetwork recovery
    true_sub = collapse_to_subnetworks(true_patterns)
    aa_sub = collapse_to_subnetworks(S_group)
    pca_sub = collapse_to_subnetworks(pca.components_)

    aa_sub_match = optimal_match(true_sub, aa_sub, absolute=False)
    pca_sub_match = optimal_match(true_sub, pca_sub, absolute=True)

    # PCA-subspace control
    true_pca_recon = pca.inverse_transform(pca.transform(true_patterns))
    pca_subspace_r = np.array([
        np.corrcoef(true_patterns[k], true_pca_recon[k])[0, 1]
        for k in range(true_patterns.shape[0])
    ])

    aa_mix = dimension_mixing_metrics(true_patterns, S_group)
    pca_mix = dimension_mixing_metrics(true_patterns, pca.components_)

    row = {
        "regime": regime,
        "rep": int(rep),
        "seed": int(seed),
        "K_fit": int(K_fit),

        "aa_mse": float(aa_mse),
        "pca_mse": float(pca_mse),

        "aa_motif_recovery": float(np.mean(aa_match["matched"])),
        "pca_motif_recovery": float(np.mean(pca_match["matched"])),

        "aa_subnetwork_recovery": float(np.mean(aa_sub_match["matched"])),
        "pca_subnetwork_recovery": float(np.mean(pca_sub_match["matched"])),

        "pca_subspace_recovery": float(np.mean(pca_subspace_r)),

        "aa_purity": aa_mix["purity"],
        "pca_purity": pca_mix["purity"],
        "aa_selectivity": aa_mix["selectivity"],
        "pca_selectivity": pca_mix["selectivity"],
        "aa_mixing_ratio": aa_mix["mixing_ratio"],
        "pca_mixing_ratio": pca_mix["mixing_ratio"],

        "aa_varexpl": float(msaa["varexpl"]),
        "pca_explained_variance": float(np.sum(pca.explained_variance_ratio_)),
        "msaa_elapsed_sec": float(msaa["elapsed"]),
    }

    if extra:
        row.update(extra)

    return row



# REGIME 1 — PCA-favorable control

Latent factors are orthogonal, continuous, Gaussian, and combined linearly. PCA should have the clearest advantage here.


In [ ]:

def simulate_pca_control(seed):
    rng = np.random.default_rng(seed)

    Q, _ = np.linalg.qr(
        rng.normal(size=(V, N_TRUE_MOTIFS))
    )
    true_factors = Q.T

    subjects = []

    for s in range(N_SUBJECTS):
        scores = rng.normal(
            size=(T_PER_SUBJECT, N_TRUE_MOTIFS)
        )

        X = scores @ true_factors
        X += rng.normal(
            0,
            BASE_NOISE_SD,
            size=X.shape,
        )
        subjects.append(X)

    return subjects, true_factors



# REGIME 2 — AA-favorable control

Each observation is a sparse convex mixture of a small number of nonorthogonal extremal motifs. This is intentionally close to archetypal geometry and serves as the positive control for MS-AA.


In [ ]:

def simulate_aa_control(seed):
    rng = np.random.default_rng(seed)
    subjects = []

    for s in range(N_SUBJECTS):
        multiplicative = (
            1.0
            + rng.normal(
                0,
                BASE_MULTIPLICATIVE_SD,
                size=TRUE_MOTIFS_RAW.shape,
            )
        )

        additive = rng.normal(
            0,
            BASE_ADDITIVE_SD,
            size=TRUE_MOTIFS_RAW.shape,
        )

        subj_motifs = TRUE_MOTIFS_RAW * multiplicative + additive

        W = rng.dirichlet(
            np.full(N_TRUE_MOTIFS, DIRICHLET_ALPHA),
            size=T_PER_SUBJECT,
        )

        X = W @ subj_motifs
        X += rng.normal(
            0,
            BASE_NOISE_SD,
            size=X.shape,
        )

        subjects.append(X)

    return subjects, TRUE_MOTIFS_RAW



# REGIME 3 — Brain-inspired metastable network dynamics

This regime is deliberately **not** generated exactly by PCA or AA.

Each subject has recurrent network configurations that persist for variable dwell times. State transitions are smoothed, multiple motifs can be simultaneously recruited, background motif activity is temporally autocorrelated, spatial motifs vary across subjects, and measurement noise is added.

The generative form is

\[
X_t = \sum_k A_{tk} M_k^{(s)} + \epsilon_t,
\]

but the latent activations \(A_{tk}\) arise from a persistent dynamical process rather than independent convex draws.


In [ ]:

def sample_state_sequence(T, K, rng, mean_dwell=MEAN_DWELL):
    states = []
    current = int(rng.integers(K))

    while len(states) < T:
        dwell = max(
            2,
            int(rng.geometric(1.0 / mean_dwell))
        )
        states.extend([current] * dwell)

        candidates = [k for k in range(K) if k != current]
        current = int(rng.choice(candidates))

    return np.asarray(states[:T], dtype=int)


def smooth_state_activations(states, K, transition_width=TRANSITION_WIDTH):
    T = len(states)
    A = np.zeros((T, K), dtype=float)
    A[np.arange(T), states] = 1.0

    if transition_width > 0:
        kernel_x = np.arange(-transition_width, transition_width + 1)
        sigma = max(1.0, transition_width / 2)
        kernel = np.exp(-(kernel_x ** 2) / (2 * sigma ** 2))
        kernel /= kernel.sum()

        for k in range(K):
            A[:, k] = np.convolve(A[:, k], kernel, mode="same")

    A /= A.sum(axis=1, keepdims=True) + 1e-12
    return A


def add_correlated_secondary_recruitment(A, rng):
    T, K = A.shape

    background = np.zeros((T, K), dtype=float)
    eps = rng.normal(0, 1, size=(T, K))

    for t in range(1, T):
        background[t] = (
            AR1_RHO * background[t - 1]
            + np.sqrt(1 - AR1_RHO ** 2) * eps[t]
        )

    # positive, smoothly varying coactivation
    background = np.exp(background)
    background /= background.sum(axis=1, keepdims=True) + 1e-12

    mixed = (
        (1.0 - SECONDARY_WEIGHT) * A
        + SECONDARY_WEIGHT * background
    )

    mixed /= mixed.sum(axis=1, keepdims=True) + 1e-12
    return mixed


def simulate_metastable(
    seed,
    noise_sd=BASE_NOISE_SD,
    heterogeneity_scale=1.0,
):
    rng = np.random.default_rng(seed)

    subjects = []
    state_sequences = []
    activations = []

    for s in range(N_SUBJECTS):
        multiplicative = (
            1.0
            + rng.normal(
                0,
                BASE_MULTIPLICATIVE_SD * heterogeneity_scale,
                size=TRUE_MOTIFS_RAW.shape,
            )
        )

        additive = rng.normal(
            0,
            BASE_ADDITIVE_SD * heterogeneity_scale,
            size=TRUE_MOTIFS_RAW.shape,
        )

        subj_motifs = TRUE_MOTIFS_RAW * multiplicative + additive

        states = sample_state_sequence(
            T_PER_SUBJECT,
            N_TRUE_MOTIFS,
            rng,
        )

        A_state = smooth_state_activations(
            states,
            N_TRUE_MOTIFS,
        )

        A = add_correlated_secondary_recruitment(
            A_state,
            rng,
        )

        X = A @ subj_motifs
        X += rng.normal(
            0,
            noise_sd,
            size=X.shape,
        )

        subjects.append(X)
        state_sequences.append(states)
        activations.append(A)

    return {
        "subjects": subjects,
        "true_patterns": TRUE_MOTIFS_RAW,
        "states": state_sequences,
        "activations": activations,
    }


## 5. Metastable-process diagnostics

In [ ]:

def metastable_diagnostics(sim):
    dwell_lengths = []

    for states in sim["states"]:
        start = 0

        for t in range(1, len(states) + 1):
            if t == len(states) or states[t] != states[start]:
                dwell_lengths.append(t - start)
                start = t

    lag1 = []

    for A in sim["activations"]:
        for k in range(A.shape[1]):
            lag1.append(
                np.corrcoef(
                    A[:-1, k],
                    A[1:, k],
                )[0, 1]
            )

    return {
        "mean_dwell": float(np.mean(dwell_lengths)),
        "median_dwell": float(np.median(dwell_lengths)),
        "mean_activation_lag1": float(np.nanmean(lag1)),
    }


demo = simulate_metastable(RNG_SEED)
print(metastable_diagnostics(demo))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(demo["activations"][0])
ax.set_xlabel("Time")
ax.set_ylabel("Motif activation")
ax.set_title("Example metastable latent activation trajectories")
plt.show()


## 6. Restart-safe checkpoints

In [ ]:

def load_results():
    if RESULTS_CSV.exists():
        df = pd.read_csv(RESULTS_CSV)
        print(f"Loaded {len(df)} completed simulations")
        return df
    return pd.DataFrame()


def is_complete(df, regime, condition, rep):
    if df.empty:
        return False

    mask = (
        (df["regime"] == regime)
        & (df["condition"] == condition)
        & (df["rep"].astype(int) == int(rep))
    )
    return bool(mask.any())


def save_row(df, row):
    new = pd.DataFrame([row])

    out = new if df.empty else pd.concat(
        [df, new],
        ignore_index=True,
    )

    out = out.drop_duplicates(
        subset=["regime", "condition", "rep"],
        keep="last",
    )

    tmp = RESULTS_CSV.with_suffix(".tmp.csv")
    out.to_csv(tmp, index=False)
    os.replace(tmp, RESULTS_CSV)

    return out


# PART A — Core three-regime benchmark

In [ ]:

results_df = load_results()

# PCA control
for rep in range(N_REPS_CONTROL):
    regime = "PCA_control"
    condition = "base"

    if is_complete(results_df, regime, condition, rep):
        print(f"SKIP {regime} rep={rep}")
        continue

    seed = int(
        np.random.SeedSequence(
            [RNG_SEED, 101, rep]
        ).generate_state(1)[0]
    )

    subjects, true_patterns = simulate_pca_control(seed)

    row = evaluate_methods(
        subjects,
        true_patterns,
        K_fit=N_TRUE_MOTIFS,
        seed=seed,
        regime=regime,
        rep=rep,
        extra={"condition": condition},
    )

    results_df = save_row(results_df, row)


# AA control
for rep in range(N_REPS_CONTROL):
    regime = "AA_control"
    condition = "base"

    if is_complete(results_df, regime, condition, rep):
        print(f"SKIP {regime} rep={rep}")
        continue

    seed = int(
        np.random.SeedSequence(
            [RNG_SEED, 202, rep]
        ).generate_state(1)[0]
    )

    subjects, true_patterns = simulate_aa_control(seed)

    row = evaluate_methods(
        subjects,
        true_patterns,
        K_fit=N_TRUE_MOTIFS,
        seed=seed,
        regime=regime,
        rep=rep,
        extra={"condition": condition},
    )

    results_df = save_row(results_df, row)


# Metastable regime
for rep in range(N_REPS_METASTABLE):
    regime = "metastable"
    condition = "base"

    if is_complete(results_df, regime, condition, rep):
        print(f"SKIP {regime} rep={rep}")
        continue

    seed = int(
        np.random.SeedSequence(
            [RNG_SEED, 303, rep]
        ).generate_state(1)[0]
    )

    sim = simulate_metastable(seed)
    diag = metastable_diagnostics(sim)

    row = evaluate_methods(
        sim["subjects"],
        sim["true_patterns"],
        K_fit=N_TRUE_MOTIFS,
        seed=seed,
        regime=regime,
        rep=rep,
        extra={
            "condition": condition,
            "mean_dwell": diag["mean_dwell"],
            "mean_activation_lag1": diag["mean_activation_lag1"],
        },
    )

    results_df = save_row(results_df, row)

print("Core benchmark complete.")


## 7. Core summary

In [ ]:

core = results_df[
    results_df["regime"].isin(
        ["PCA_control", "AA_control", "metastable"]
    )
].copy()

summary = (
    core
    .groupby("regime")[
        [
            "aa_motif_recovery",
            "pca_motif_recovery",
            "pca_subspace_recovery",
            "aa_subnetwork_recovery",
            "pca_subnetwork_recovery",
            "aa_purity",
            "pca_purity",
            "aa_mse",
            "pca_mse",
        ]
    ]
    .agg(["mean", "sem"])
)

display(summary)


In [ ]:

regimes = ["PCA_control", "AA_control", "metastable"]
labels = ["PCA control", "AA control", "Metastable"]

plot_df = (
    core
    .groupby("regime")[
        ["aa_motif_recovery", "pca_motif_recovery"]
    ]
    .agg(["mean", "sem"])
)

x = np.arange(len(regimes))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4.5))

aa_means = [
    plot_df.loc[r, ("aa_motif_recovery", "mean")]
    for r in regimes
]
aa_sems = [
    plot_df.loc[r, ("aa_motif_recovery", "sem")]
    for r in regimes
]

pca_means = [
    plot_df.loc[r, ("pca_motif_recovery", "mean")]
    for r in regimes
]
pca_sems = [
    plot_df.loc[r, ("pca_motif_recovery", "sem")]
    for r in regimes
]

ax.bar(
    x - width / 2,
    aa_means,
    width,
    yerr=aa_sems,
    label="MS-AA",
)

ax.bar(
    x + width / 2,
    pca_means,
    width,
    yerr=pca_sems,
    label="PCA component",
)

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("One-to-one latent-pattern recovery")
ax.set_ylim(0, 1.05)
ax.set_title("Method-specific controls and brain-inspired dynamics")
ax.legend()

plt.show()


In [ ]:

subspace = (
    core
    .groupby("regime")["pca_subspace_recovery"]
    .agg(["mean", "sem"])
)

fig, ax = plt.subplots(figsize=(8, 4.5))

means = [subspace.loc[r, "mean"] for r in regimes]
sems = [subspace.loc[r, "sem"] for r in regimes]

ax.errorbar(
    x,
    means,
    yerr=sems,
    marker="o",
)

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("PCA-subspace recovery")
ax.set_ylim(0, 1.05)
ax.set_title("Latent information retained in the PCA subspace")
plt.show()


# PART B — Metastable robustness to measurement noise

In [ ]:

results_df = load_results()

for noise_sd in NOISE_LEVELS:
    for rep in range(N_REPS_ROBUST):
        regime = "metastable_noise"
        condition = f"noise_{noise_sd}"

        if is_complete(results_df, regime, condition, rep):
            print(f"SKIP {condition}, rep={rep}")
            continue

        seed = int(
            np.random.SeedSequence(
                [RNG_SEED, 404, int(noise_sd * 1000), rep]
            ).generate_state(1)[0]
        )

        sim = simulate_metastable(
            seed,
            noise_sd=noise_sd,
            heterogeneity_scale=1.0,
        )

        diag = metastable_diagnostics(sim)

        row = evaluate_methods(
            sim["subjects"],
            sim["true_patterns"],
            K_fit=N_TRUE_MOTIFS,
            seed=seed,
            regime=regime,
            rep=rep,
            extra={
                "condition": condition,
                "noise_sd": noise_sd,
                "mean_dwell": diag["mean_dwell"],
                "mean_activation_lag1": diag["mean_activation_lag1"],
            },
        )

        results_df = save_row(results_df, row)

print("Noise robustness complete.")


# PART C — Metastable robustness to subject heterogeneity

In [ ]:

results_df = load_results()

for hetero in HETEROGENEITY_LEVELS:
    for rep in range(N_REPS_ROBUST):
        regime = "metastable_heterogeneity"
        condition = f"heterogeneity_{hetero}"

        if is_complete(results_df, regime, condition, rep):
            print(f"SKIP {condition}, rep={rep}")
            continue

        seed = int(
            np.random.SeedSequence(
                [RNG_SEED, 505, int(hetero * 1000), rep]
            ).generate_state(1)[0]
        )

        sim = simulate_metastable(
            seed,
            noise_sd=BASE_NOISE_SD,
            heterogeneity_scale=hetero,
        )

        diag = metastable_diagnostics(sim)

        row = evaluate_methods(
            sim["subjects"],
            sim["true_patterns"],
            K_fit=N_TRUE_MOTIFS,
            seed=seed,
            regime=regime,
            rep=rep,
            extra={
                "condition": condition,
                "heterogeneity_scale": hetero,
                "mean_dwell": diag["mean_dwell"],
                "mean_activation_lag1": diag["mean_activation_lag1"],
            },
        )

        results_df = save_row(results_df, row)

print("Heterogeneity robustness complete.")


# PART D — Metastable sensitivity to fitted model order K

In [ ]:

results_df = load_results()

for K_fit in K_LEVELS:
    for rep in range(N_REPS_ROBUST):
        regime = "metastable_K"
        condition = f"K_{K_fit}"

        if is_complete(results_df, regime, condition, rep):
            print(f"SKIP {condition}, rep={rep}")
            continue

        seed = int(
            np.random.SeedSequence(
                [RNG_SEED, 606, int(K_fit), rep]
            ).generate_state(1)[0]
        )

        sim = simulate_metastable(seed)
        diag = metastable_diagnostics(sim)

        row = evaluate_methods(
            sim["subjects"],
            sim["true_patterns"],
            K_fit=K_fit,
            seed=seed,
            regime=regime,
            rep=rep,
            extra={
                "condition": condition,
                "mean_dwell": diag["mean_dwell"],
                "mean_activation_lag1": diag["mean_activation_lag1"],
            },
        )

        results_df = save_row(results_df, row)

print("K sensitivity complete.")


# PART E — Representative metastable visualization

In [ ]:

REPRESENTATIVE_SEED = 20260908

sim = simulate_metastable(REPRESENTATIVE_SEED)
subjects, mu, sd = normalize_across_subjects(sim["subjects"])
true_patterns = (sim["true_patterns"] - mu) / sd

msaa = fit_msaa(
    subjects,
    K=N_TRUE_MOTIFS,
    analysis_type="spatial",
    seed=REPRESENTATIVE_SEED,
    verbose=True,
)

S_group = np.mean(
    np.stack(
        [np.asarray(sub["S"], dtype=float) for sub in msaa["results_subj"]],
        axis=0,
    ),
    axis=0,
)

X_stack = np.vstack(subjects)

pca = PCA(
    n_components=N_TRUE_MOTIFS,
    random_state=REPRESENTATIVE_SEED,
).fit(X_stack)

aa_match = optimal_match(
    true_patterns,
    S_group,
    absolute=False,
)

pca_match = optimal_match(
    true_patterns,
    pca.components_,
    absolute=True,
)

AA_ORDERED = S_group[aa_match["recovered_idx"]].copy()
PCA_ORDERED = pca.components_[pca_match["recovered_idx"]].copy()

for i in range(len(PCA_ORDERED)):
    if np.corrcoef(true_patterns[i], PCA_ORDERED[i])[0, 1] < 0:
        PCA_ORDERED[i] *= -1

plot_pattern_matrix(
    true_patterns,
    "True recurring network configurations",
    ylabel="True configuration",
)

plot_pattern_matrix(
    AA_ORDERED,
    "Matched MS-AA archetypes",
    ylabel="Matched archetype",
)

plot_pattern_matrix(
    PCA_ORDERED,
    "Matched PCA components",
    ylabel="Matched component",
)


In [ ]:

def plot_similarity_matrix(C, title):
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    im = ax.imshow(C, vmin=0, vmax=1, aspect="auto")
    ax.set_xlabel("Recovered dimension")
    ax.set_ylabel("True configuration")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label="Absolute pattern similarity")
    plt.show()


plot_similarity_matrix(
    np.abs(pattern_corr_matrix(true_patterns, S_group)),
    "True configurations ↔ MS-AA archetypes",
)

plot_similarity_matrix(
    pattern_corr_matrix(true_patterns, pca.components_, absolute=True),
    "True configurations ↔ PCA components",
)


# PART F — Final manuscript-oriented summary

In [ ]:

results_df = load_results()

core = results_df[
    results_df["regime"].isin(
        ["PCA_control", "AA_control", "metastable"]
    )
].copy()

final_rows = []

for regime in ["PCA_control", "AA_control", "metastable"]:
    sub = core[core["regime"] == regime]

    if len(sub) == 0:
        continue

    final_rows.append({
        "regime": regime,
        "n": len(sub),
        "AA motif recovery": sub["aa_motif_recovery"].mean(),
        "PCA component recovery": sub["pca_motif_recovery"].mean(),
        "AA - PCA recovery": (
            sub["aa_motif_recovery"]
            - sub["pca_motif_recovery"]
        ).mean(),
        "PCA subspace recovery": sub["pca_subspace_recovery"].mean(),
        "AA subnetwork recovery": sub["aa_subnetwork_recovery"].mean(),
        "PCA subnetwork recovery": sub["pca_subnetwork_recovery"].mean(),
        "AA MSE": sub["aa_mse"].mean(),
        "PCA MSE": sub["pca_mse"].mean(),
        "AA purity": sub["aa_purity"].mean(),
        "PCA purity": sub["pca_purity"].mean(),
    })

final_summary_df = pd.DataFrame(final_rows)
display(final_summary_df)



# Interpretation guide

The ideal result is a **three-way calibration**:

### PCA-favorable control
PCA should recover the true orthogonal Gaussian factors at least as well as MS-AA and should have the strongest reconstruction fidelity. This shows the benchmark does not intrinsically favor archetypes.

### AA-favorable control
MS-AA should more directly recover the true extremal motifs. PCA may still represent those motifs accurately in its full low-dimensional subspace, even if its individual components are rotated mixtures.

### Brain-inspired metastable regime
This is the scientifically important test. The data are not generated exactly by either method. If PCA retains the latent information in its subspace while MS-AA provides stronger one-to-one recovery of recurring network configurations and their subnetwork composition, that gives us a principled reason to use archetypes for interpretation.

The manuscript claim should remain narrow:

> **The simulation does not assume that brain activity is literally generated by a simplex model. Rather, it shows that under temporally persistent, recurrent, overlapping, subject-variable network dynamics, MS-AA can provide a more direct representation of underlying network configurations than PCA, even when PCA preserves the same information within a low-dimensional subspace.**



# PART G — Paired uncertainty for the MS-AA vs PCA recovery difference

Because MS-AA and PCA are evaluated on the **same simulated dataset within each replicate**, the primary inferential quantity is the paired difference

\[
\Delta r = r_{\mathrm{MS-AA}} - r_{\mathrm{PCA}}.
\]

This section adds:

- the mean paired difference for each generative regime,
- a nonparametric bootstrap 95% confidence interval across simulation replicates,
- the proportion of replicates in which the difference has the expected sign,
- a two-sided sign-flip permutation test against a null mean difference of zero,
- the same paired analysis for subnetwork recovery,
- a manuscript-ready plot of replicate-level differences.

These analyses use the checkpointed simulation results and **do not require rerunning the MS-AA fits**.


In [ ]:

# ============================================================
# Paired bootstrap + sign-flip inference
# ============================================================

from scipy.stats import ttest_1samp

PAIR_BOOT_REPS = 20000
PAIR_PERM_REPS = 20000
PAIR_RNG_SEED = 20260908

results_df = load_results()

core = results_df[
    results_df["regime"].isin(
        ["PCA_control", "AA_control", "metastable"]
    )
].copy()

EXPECTED_SIGN = {
    "PCA_control": -1,   # PCA should recover the true factors better
    "AA_control": +1,    # MS-AA should recover extremal motifs better
    "metastable": +1,    # empirical question of interest
}


def bootstrap_mean_ci(values, n_boot=PAIR_BOOT_REPS, seed=PAIR_RNG_SEED):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    rng = np.random.default_rng(seed)

    idx = rng.integers(
        0,
        len(values),
        size=(n_boot, len(values)),
    )

    boot_means = values[idx].mean(axis=1)

    lo, hi = np.quantile(
        boot_means,
        [0.025, 0.975],
    )

    return float(values.mean()), float(lo), float(hi)


def signflip_pvalue(values, n_perm=PAIR_PERM_REPS, seed=PAIR_RNG_SEED):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    observed = abs(values.mean())

    rng = np.random.default_rng(seed)

    signs = rng.choice(
        [-1.0, 1.0],
        size=(n_perm, len(values)),
    )

    perm_means = np.abs(
        (signs * values[None, :]).mean(axis=1)
    )

    p = (
        np.sum(perm_means >= observed) + 1
    ) / (
        n_perm + 1
    )

    return float(p)


paired_rows = []

for i, regime in enumerate(
    ["PCA_control", "AA_control", "metastable"]
):
    sub = (
        core[core["regime"] == regime]
        .sort_values("rep")
        .copy()
    )

    motif_delta = (
        sub["aa_motif_recovery"].to_numpy()
        - sub["pca_motif_recovery"].to_numpy()
    )

    subnet_delta = (
        sub["aa_subnetwork_recovery"].to_numpy()
        - sub["pca_subnetwork_recovery"].to_numpy()
    )

    motif_mean, motif_lo, motif_hi = bootstrap_mean_ci(
        motif_delta,
        seed=PAIR_RNG_SEED + i,
    )

    subnet_mean, subnet_lo, subnet_hi = bootstrap_mean_ci(
        subnet_delta,
        seed=PAIR_RNG_SEED + 100 + i,
    )

    expected_sign = EXPECTED_SIGN[regime]

    paired_rows.append({
        "regime": regime,
        "n_replicates": len(sub),

        "mean_delta_motif": motif_mean,
        "motif_CI_low": motif_lo,
        "motif_CI_high": motif_hi,
        "motif_signflip_p": signflip_pvalue(
            motif_delta,
            seed=PAIR_RNG_SEED + 200 + i,
        ),
        "motif_expected_sign_fraction": float(
            np.mean(np.sign(motif_delta) == expected_sign)
        ),

        "mean_delta_subnetwork": subnet_mean,
        "subnetwork_CI_low": subnet_lo,
        "subnetwork_CI_high": subnet_hi,
        "subnetwork_signflip_p": signflip_pvalue(
            subnet_delta,
            seed=PAIR_RNG_SEED + 300 + i,
        ),
        "subnetwork_expected_sign_fraction": float(
            np.mean(np.sign(subnet_delta) == expected_sign)
        ),
    })

paired_inference_df = pd.DataFrame(paired_rows)

display(paired_inference_df)


In [ ]:

# ============================================================
# Replicate-level paired differences
# ============================================================

paired_long_rows = []

for regime in ["PCA_control", "AA_control", "metastable"]:
    sub = (
        core[core["regime"] == regime]
        .sort_values("rep")
        .copy()
    )

    for _, row in sub.iterrows():
        paired_long_rows.append({
            "regime": regime,
            "rep": int(row["rep"]),
            "delta_motif": (
                row["aa_motif_recovery"]
                - row["pca_motif_recovery"]
            ),
            "delta_subnetwork": (
                row["aa_subnetwork_recovery"]
                - row["pca_subnetwork_recovery"]
            ),
            "delta_mse": (
                row["aa_mse"]
                - row["pca_mse"]
            ),
        })

paired_long_df = pd.DataFrame(paired_long_rows)

display(
    paired_long_df
    .groupby("regime")[
        ["delta_motif", "delta_subnetwork", "delta_mse"]
    ]
    .agg(["mean", "std", "min", "max"])
)


In [ ]:

# ============================================================
# Manuscript-ready paired-difference plot
# ============================================================

regimes = ["PCA_control", "AA_control", "metastable"]
labels = ["PCA-favorable", "AA-favorable", "Metastable"]

fig, ax = plt.subplots(figsize=(8, 4.8))

x = np.arange(len(regimes))

for i, regime in enumerate(regimes):
    vals = paired_long_df.loc[
        paired_long_df["regime"] == regime,
        "delta_motif",
    ].to_numpy()

    jitter_rng = np.random.default_rng(PAIR_RNG_SEED + i)
    jitter = jitter_rng.normal(
        0,
        0.035,
        size=len(vals),
    )

    ax.scatter(
        np.full(len(vals), x[i]) + jitter,
        vals,
        alpha=0.65,
        s=30,
    )

    row = paired_inference_df[
        paired_inference_df["regime"] == regime
    ].iloc[0]

    mean = row["mean_delta_motif"]
    lo = row["motif_CI_low"]
    hi = row["motif_CI_high"]

    ax.errorbar(
        x[i],
        mean,
        yerr=np.array([[mean - lo], [hi - mean]]),
        fmt="o",
        markersize=8,
        capsize=5,
        linewidth=2,
    )

ax.axhline(0, linestyle="--", linewidth=1)

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel(
    r"Paired motif-recovery difference, "
    r"$\Delta r = r_{MS-AA} - r_{PCA}$"
)
ax.set_title(
    "Method advantage reverses across generative controls "
    "and favors MS-AA for metastable motifs"
)

plt.show()



## How to report this result

The paired difference is the cleanest primary simulation statistic because the two methods are applied to the **same realization** of each simulated dataset.

For the two control regimes, the important result is that the sign of the method advantage reverses as expected:

- \(\Delta r < 0\) for the PCA-favorable Gaussian-factor control;
- \(\Delta r > 0\) for the AA-favorable extremal-mixture control.

For the brain-inspired metastable regime, the primary question is whether the bootstrap confidence interval for \(\Delta r\) remains above zero. If so, this supports the interpretation that MS-AA more directly recovers recurring latent network configurations, even though PCA may still reconstruct the observations better and preserve the same latent information in its full subspace.

The sign-flip permutation test is included as a complementary nonparametric test. For the manuscript, the **effect size and bootstrap CI should be emphasized more strongly than the p-value**.
